# Forward Pass MLP dari Nol

**Tujuan Project:**
Memahami bahwa **fungsi aktivasi nonlinear** adalah komponen yang membuat neural network
lebih ekspresif dibanding sekadar rangkaian transformasi linear. Pemahaman ini adalah
fondasi penting sebelum mempelajari LSTM, karena setiap *gate* pada LSTM (forget gate,
input gate, output gate, candidate cell state) selalu dibungkus oleh fungsi aktivasi
nonlinear — umumnya **sigmoid (σ)** untuk gate (nilai 0–1, berperan sebagai "saklar"/filter)
dan **tanh** untuk kandidat nilai (nilai -1–1, berperan membawa informasi).

**Yang dibangun di notebook ini:**
- MLP dengan **1 hidden layer**: `4 input → 8 hidden (tanh) → 2 output`
- Hanya **forward pass** (tidak ada training/backpropagation)
- Perbandingan: MLP **dengan** aktivasi nonlinear vs MLP **tanpa** aktivasi (murni linear)
- Pembuktian matematis + eksperimen bahwa MLP tanpa aktivasi nonlinear akan **runtuh
  (collapse)** menjadi satu transformasi linear tunggal, tidak peduli berapa banyak layer
  yang ditumpuk.

Semua dibangun **dari nol** menggunakan `numpy` saja (tanpa framework deep learning),
supaya setiap operasi matriks terlihat eksplisit.


## 1. Teori Dasar: Anatomi Satu Layer MLP

Satu *layer* (lapisan) pada MLP melakukan dua tahap operasi terhadap input $x$:

1. **Transformasi affine (linear):**
$$z = xW + b$$
di mana $W$ adalah matriks bobot (*weight*) dan $b$ adalah *bias*. Operasi ini murni
linear — hanya melakukan rotasi, penskalaan, dan pergeseran ruang input.

2. **Fungsi aktivasi (nonlinear):**
$$a = f(z)$$
di mana $f$ adalah fungsi nonlinear seperti `tanh`, `sigmoid`, atau `ReLU`. Tahap inilah
yang **melengkungkan** ruang representasi sehingga network bisa memodelkan hubungan yang
tidak bisa direpresentasikan oleh garis/bidang lurus.

Untuk network `4 → 8 → 2` yang kita bangun:

$$
\begin{aligned}
Z_1 &= X W_1 + b_1 \quad &\text{(4 → 8)} \\
A_1 &= \tanh(Z_1) \quad &\text{(aktivasi hidden layer)} \\
Z_2 &= A_1 W_2 + b_2 \quad &\text{(8 → 2, output layer)}
\end{aligned}
$$

Di sini $Z_2$ langsung dipakai sebagai output (regresi, tanpa aktivasi output karena kita
akan membandingkan dengan target kontinu memakai **MSE loss**):

$$
\text{MSE}(\hat{y}, y) = \frac{1}{N}\sum_{i=1}^{N} (\hat{y}_i - y_i)^2
$$

**Fungsi `tanh`** dipilih di sini karena bentuknya sama dengan yang dipakai pada
kandidat *cell state* LSTM ($\tilde{C}_t = \tanh(\dots)$), sehingga pemahaman perilakunya
langsung relevan untuk materi LSTM selanjutnya. Bentuknya:

$$\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}, \quad \text{range: } (-1, 1)$$


## 2. Setup

Import `numpy` untuk operasi matriks, dan set random seed agar hasil eksperimen dapat
direproduksi (reproducible).

In [1]:
import numpy as np

# Seed agar hasil random selalu sama setiap kali notebook dijalankan ulang
SEED = 42
rng = np.random.default_rng(SEED)

# Arsitektur network: 4 input -> 8 hidden -> 2 output
N_IN = 4
N_HID = 8
N_OUT = 2

# Ukuran batch data acak yang akan kita forward-pass-kan
BATCH_SIZE = 5

print(f"Arsitektur MLP: {N_IN} -> {N_HID} (tanh) -> {N_OUT}")
print(f"Batch size    : {BATCH_SIZE}")


Arsitektur MLP: 4 -> 8 (tanh) -> 2
Batch size    : 5


## 3. Inisialisasi Bobot Acak

Karena kita hanya melakukan **forward pass** (tanpa training), bobot cukup diinisialisasi
secara acak dari distribusi normal dengan standar deviasi kecil (`0.5` untuk $W$, `0.1`
untuk $b$) agar nilai pra-aktivasi ($z$) tidak terlalu besar/kecil (menghindari saturasi
`tanh` yang ekstrem sejak awal).

Bentuk (*shape*) matriks bobot mengikuti aturan perkalian matriks $X W$:

| Parameter | Shape           | Keterangan                          |
|-----------|-----------------|--------------------------------------|
| $W_1$     | (4, 8)          | menghubungkan 4 input ke 8 hidden    |
| $b_1$     | (8,)            | bias untuk 8 neuron hidden           |
| $W_2$     | (8, 2)          | menghubungkan 8 hidden ke 2 output   |
| $b_2$     | (2,)            | bias untuk 2 neuron output           |


In [2]:
def init_weights(n_in: int, n_hid: int, n_out: int, rng: np.random.Generator):
        # Inisialisasi bobot & bias acak untuk MLP 1 hidden layer.
    # Returns
    # -------
    # W1 : (n_in, n_hid)
    # b1 : (n_hid,)
    # W2 : (n_hid, n_out)
    # b2 : (n_out,)
    W1 = rng.normal(loc=0.0, scale=0.5, size=(n_in, n_hid))
    b1 = rng.normal(loc=0.0, scale=0.1, size=(n_hid,))
    W2 = rng.normal(loc=0.0, scale=0.5, size=(n_hid, n_out))
    b2 = rng.normal(loc=0.0, scale=0.1, size=(n_out,))
    return W1, b1, W2, b2


W1, b1, W2, b2 = init_weights(N_IN, N_HID, N_OUT, rng)

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)


W1 shape: (4, 8)
b1 shape: (8,)
W2 shape: (8, 2)
b2 shape: (2,)


## 4. Data Batch Acak & Target Acak

Kita bangkitkan `X` (input) dan `Y_target` (target) secara acak dari distribusi normal
standar. Karena tidak ada training, target ini semata-mata dipakai untuk menghitung
**MSE loss** sebagai gambaran seberapa jauh output awal (belum dilatih) dari target —
bukan untuk dioptimasi.

In [3]:
X = rng.normal(loc=0.0, scale=1.0, size=(BATCH_SIZE, N_IN))
Y_target = rng.normal(loc=0.0, scale=1.0, size=(BATCH_SIZE, N_OUT))

print("X (input batch), shape", X.shape, ":\n", X)
print("\nY_target (target acak), shape", Y_target.shape, ":\n", Y_target)


X (input batch), shape (5, 4) :
 [[-0.86583112  0.96827835 -1.68286977 -0.33488503]
 [ 0.16275307  0.58622233  0.71122658  0.79334724]
 [-0.34872507 -0.46235179  0.85797588 -0.19130432]
 [-1.27568632 -1.13328721 -0.91945229  0.49716074]
 [ 0.14242574  0.69048535 -0.42725265  0.15853969]]

Y_target (target acak), shape (5, 2) :
 [[ 0.62559039 -0.30934654]
 [ 0.45677524 -0.66192594]
 [-0.36305385 -0.38173789]
 [-1.19583965  0.48697248]
 [-0.46940234  0.01249412]]


## 5. Forward Pass DENGAN Aktivasi Nonlinear (tanh)

Implementasi persis mengikuti formulasi di bagian teori:

$$Z_1 = XW_1 + b_1 \;\rightarrow\; A_1 = \tanh(Z_1) \;\rightarrow\; Z_2 = A_1 W_2 + b_2 = \hat{y}$$

Fungsi MSE loss juga didefinisikan dari nol (tanpa library ML).

In [4]:
def mse_loss(pred: np.ndarray, target: np.ndarray) -> float:
        # Mean Squared Error antara prediksi dan target.
    return float(np.mean((pred - target) ** 2))


def forward_nonlinear(X, W1, b1, W2, b2):
        # Forward pass MLP 1 hidden layer DENGAN aktivasi tanh di hidden layer.
    Z1 = X @ W1 + b1          # (batch, 8) -- transformasi linear ke hidden
    A1 = np.tanh(Z1)          # (batch, 8) -- NONLINEARITAS di sini
    Z2 = A1 @ W2 + b2         # (batch, 2) -- transformasi linear ke output
    return Z2, A1, Z1


y_pred_nonlinear, A1, Z1 = forward_nonlinear(X, W1, b1, W2, b2)
loss_nonlinear = mse_loss(y_pred_nonlinear, Y_target)

print("Pra-aktivasi hidden layer Z1 (contoh baris pertama):\n", Z1[0])
print("\nSetelah tanh, A1 (contoh baris pertama, semua nilai terjepit di (-1, 1)):\n", A1[0])
print("\nOutput prediksi (dengan aktivasi nonlinear):\n", y_pred_nonlinear)
print("\nMSE Loss (nonlinear):", loss_nonlinear)


Pra-aktivasi hidden layer Z1 (contoh baris pertama):
 [-0.42983462  0.82165544 -0.66582296  0.06309021  0.95164744  1.52627987
 -1.29874442 -0.01597388]

Setelah tanh, A1 (contoh baris pertama, semua nilai terjepit di (-1, 1)):
 [-0.40518309  0.67596989 -0.58222552  0.06300663  0.74052798  0.90978582
 -0.86139958 -0.01597252]

Output prediksi (dengan aktivasi nonlinear):
 [[ 0.6047095   0.51627887]
 [-0.1070379   0.19102683]
 [-0.22293517 -0.09299455]
 [-0.30368247  0.01211375]
 [ 0.31524738  0.35407995]]

MSE Loss (nonlinear): 0.3584304267056402


## 6. Forward Pass TANPA Aktivasi (Semua Layer Linear)

Sekarang kita ulangi eksperimen yang **persis sama** (bobot, input, target sama), tapi
fungsi aktivasi `tanh` dihilangkan — diganti fungsi identitas $f(z) = z$. Dengan kata
lain, hidden layer hanya melakukan transformasi linear murni, tanpa "lengkungan" apa pun.

$$Z_1 = XW_1 + b_1 \;\rightarrow\; A_1 = Z_1 \text{ (identity, tanpa nonlinearitas)} \;\rightarrow\; Z_2 = A_1 W_2 + b_2$$


In [5]:
def forward_linear(X, W1, b1, W2, b2):
        # Forward pass MLP 1 hidden layer TANPA aktivasi (semua layer linear/identity).
    Z1 = X @ W1 + b1          # (batch, 8) -- transformasi linear ke hidden
    A1 = Z1                   # identity: TIDAK ADA nonlinearitas
    Z2 = A1 @ W2 + b2         # (batch, 2) -- transformasi linear ke output
    return Z2, A1, Z1


y_pred_linear, A1_lin, Z1_lin = forward_linear(X, W1, b1, W2, b2)
loss_linear = mse_loss(y_pred_linear, Y_target)

print("Output prediksi (TANPA aktivasi / murni linear):\n", y_pred_linear)
print("\nMSE Loss (linear):", loss_linear)


Output prediksi (TANPA aktivasi / murni linear):
 [[ 1.02165609  0.78899385]
 [-0.35530479  0.12742325]
 [-0.2315221  -0.10522861]
 [-0.40401091  0.00982725]
 [ 0.33680035  0.36535973]]

MSE Loss (linear): 0.43686609891080375


## 7. Pembuktian: MLP Linear Bertumpuk = Satu Transformasi Linear Tunggal

**Klaim:** Jika tidak ada fungsi aktivasi nonlinear di antara layer-layer, maka
menumpuk berapa pun banyak layer linear **secara matematis setara** dengan satu layer
linear tunggal — network "runtuh" (collapse), kehilangan seluruh manfaat kedalaman.

**Derivasi:**

Dari forward pass linear:

$$
Z_1 = X W_1 + b_1
$$

$$
Z_2 = Z_1 W_2 + b_2 = (X W_1 + b_1) W_2 + b_2
$$

Uraikan (distribusikan) perkalian matriks:

$$
Z_2 = X \underbrace{(W_1 W_2)}_{W_{\text{eff}}} + \underbrace{(b_1 W_2 + b_2)}_{b_{\text{eff}}}
$$

Sehingga:

$$
Z_2 = X W_{\text{eff}} + b_{\text{eff}}, \quad\text{dengan}\quad
W_{\text{eff}} = W_1 W_2 \in \mathbb{R}^{4\times2}, \quad
b_{\text{eff}} = b_1 W_2 + b_2 \in \mathbb{R}^{2}
$$

Ini persis bentuk **satu** transformasi affine/linear dari 4 input langsung ke 2 output —
sama seperti network *tanpa* hidden layer sama sekali. Argumen ini berlaku secara induktif
untuk berapa pun banyak layer linear yang ditumpuk (komposisi fungsi linear tetap linear):
kompleksitas *depth* (kedalaman) yang ditambahkan sama sekali tidak menambah **daya
ekspresi** (*expressive power*) selama tidak ada nonlinearitas di antaranya.

Konsekuensinya, `W_eff` yang berukuran (4, 2) hanya bisa punya rank maksimum 2 (dibatasi
oleh dimensi terkecil), walaupun `W1` (4x8) dan `W2` (8x2) masing-masing punya rank
lebih tinggi — informasi "hilang" karena semuanya bisa diringkas jadi satu matriks kecil.

Sekarang kita buktikan klaim ini secara eksperimen dengan `numpy`.

In [6]:
# Hitung W_eff dan b_eff sesuai hasil derivasi di atas
W_eff = W1 @ W2                 # (4, 8) @ (8, 2) -> (4, 2)
b_eff = b1 @ W2 + b2            # (8,) @ (8, 2) + (2,) -> (2,)

# Output jika kita langsung pakai satu transformasi linear tunggal
y_pred_single = X @ W_eff + b_eff

print("W_eff shape:", W_eff.shape, "  b_eff shape:", b_eff.shape)
print("\nOutput dari MLP linear 2-layer (Bagian 6):\n", y_pred_linear)
print("\nOutput dari SATU transformasi linear tunggal (W_eff, b_eff):\n", y_pred_single)

# Bandingkan keduanya -- seharusnya identik (hanya beda floating point epsilon)
identik = np.allclose(y_pred_linear, y_pred_single)
selisih_maks = np.max(np.abs(y_pred_linear - y_pred_single))

print(f"\nApakah kedua output identik (np.allclose)? {identik}")
print(f"Selisih absolut maksimum antar keduanya   : {selisih_maks:.2e}")


W_eff shape: (4, 2)   b_eff shape: (2,)

Output dari MLP linear 2-layer (Bagian 6):
 [[ 1.02165609  0.78899385]
 [-0.35530479  0.12742325]
 [-0.2315221  -0.10522861]
 [-0.40401091  0.00982725]
 [ 0.33680035  0.36535973]]

Output dari SATU transformasi linear tunggal (W_eff, b_eff):
 [[ 1.02165609  0.78899385]
 [-0.35530479  0.12742325]
 [-0.2315221  -0.10522861]
 [-0.40401091  0.00982725]
 [ 0.33680035  0.36535973]]

Apakah kedua output identik (np.allclose)? True
Selisih absolut maksimum antar keduanya   : 1.67e-16


**Interpretasi hasil di atas:**
`np.allclose` bernilai `True` dan selisih maksimum berada pada orde `1e-16` (galat
pembulatan *floating point*, bukan perbedaan matematis riil). Ini membuktikan secara
eksperimen bahwa MLP linear 2 layer (`4→8→2` tanpa aktivasi) **identik** dengan satu
transformasi linear tunggal `4→2`. Menambah hidden layer tanpa nonlinearitas hanyalah
*reparameterisasi* — bukan penambahan kapasitas model yang sesungguhnya.

Sebagai bukti tambahan, kita bandingkan **rank** matriks bobot: meskipun $W_1$ (4×8) dan
$W_2$ (8×2) masing-masing punya rank yang relatif tinggi, matriks komposisinya
$W_{\text{eff}} = W_1 W_2$ dibatasi oleh dimensi paling sempit di rantai perkalian.

In [7]:
rank_W1 = np.linalg.matrix_rank(W1)
rank_W2 = np.linalg.matrix_rank(W2)
rank_Weff = np.linalg.matrix_rank(W_eff)

print(f"Rank W1  (4x8) : {rank_W1}")
print(f"Rank W2  (8x2) : {rank_W2}")
print(f"Rank W_eff (4x2) hasil komposisi W1 @ W2 : {rank_Weff}")
print("\n-> Rank W_eff tidak bisa melebihi min(rank W1, rank W2, 4, 2) = 2.")
print("   Informasi dari 8 neuron hidden 'diringkas paksa' menjadi bottleneck rank <= 2.")


Rank W1  (4x8) : 4
Rank W2  (8x2) : 2
Rank W_eff (4x2) hasil komposisi W1 @ W2 : 2

-> Rank W_eff tidak bisa melebihi min(rank W1, rank W2, 4, 2) = 2.
   Informasi dari 8 neuron hidden 'diringkas paksa' menjadi bottleneck rank <= 2.


## 8. Perbandingan Langsung: Nonlinear vs Linear

Ringkasan side-by-side dari kedua eksperimen (bobot awal, input, dan target sama persis;
satu-satunya perbedaan adalah ada/tidaknya `tanh` di hidden layer).

In [8]:
print("=" * 70)
print(f"{'':25s}{'Dengan tanh (nonlinear)':>22s}{'Tanpa aktivasi (linear)':>23s}")
print("=" * 70)
print(f"{'MSE Loss':25s}{loss_nonlinear:>22.6f}{loss_linear:>23.6f}")
print("-" * 70)
print("Output (baris demi baris, y_pred[0], y_pred[1] per sampel):\n")
for i in range(BATCH_SIZE):
    nl = np.array2string(y_pred_nonlinear[i], precision=4, floatmode='fixed')
    ln = np.array2string(y_pred_linear[i], precision=4, floatmode='fixed')
    print(f"  sampel {i}:  nonlinear={nl}   linear={ln}")

print("\nApakah output nonlinear == output linear?",
      np.allclose(y_pred_nonlinear, y_pred_linear))
print("(Diharapkan False -- keduanya BERBEDA karena tanh melengkungkan ruang representasi)")


                         Dengan tanh (nonlinear)Tanpa aktivasi (linear)
MSE Loss                               0.358430               0.436866
----------------------------------------------------------------------
Output (baris demi baris, y_pred[0], y_pred[1] per sampel):

  sampel 0:  nonlinear=[0.6047 0.5163]   linear=[1.0217 0.7890]
  sampel 1:  nonlinear=[-0.1070  0.1910]   linear=[-0.3553  0.1274]
  sampel 2:  nonlinear=[-0.2229 -0.0930]   linear=[-0.2315 -0.1052]
  sampel 3:  nonlinear=[-0.3037  0.0121]   linear=[-0.4040  0.0098]
  sampel 4:  nonlinear=[0.3152 0.3541]   linear=[0.3368 0.3654]

Apakah output nonlinear == output linear? False
(Diharapkan False -- keduanya BERBEDA karena tanh melengkungkan ruang representasi)


## 9. Kesimpulan: Mengapa Keduanya Berbeda Secara Fundamental

1. **Komposisi fungsi linear tetap linear.** Sudah dibuktikan pada Bagian 7 bahwa
   menumpuk layer-layer linear (tanpa aktivasi nonlinear di antaranya) — berapa pun
   dalamnya — secara matematis selalu bisa diringkas menjadi **satu** matriks bobot
   efektif $W_{\text{eff}}$ dan satu bias efektif $b_{\text{eff}}$. Kedalaman (*depth*)
   network menjadi tidak berarti secara ekspresif; ia hanya "kedalaman semu".

2. **Nonlinearitas ($\tanh$) mematahkan sifat komposabilitas linear ini.** Karena
   $\tanh(x_1 + x_2) \neq \tanh(x_1) + \tanh(x_2)$ (tidak memenuhi sifat aditivitas/
   homogenitas fungsi linear), maka $A_1 = \tanh(Z_1)$ **tidak bisa** diserap kembali
   ke dalam satu perkalian matriks tunggal. Setiap layer nonlinear benar-benar menambah
   *ruang hipotesis* (kelas fungsi) yang bisa direpresentasikan network — inilah dasar
   dari **Universal Approximation Theorem**: MLP dengan minimal satu hidden layer
   berfungsi aktivasi nonlinear dapat mengaproksimasi fungsi kontinu apa pun (dengan
   neuron yang cukup), sedangkan MLP tanpa aktivasi nonlinear **selamanya** hanya bisa
   merepresentasikan fungsi linear/affine, seberapa pun dalam/lebarnya.

3. **Bukti eksperimen mendukung teori.** Nilai MSE loss dan output prediksi antara
   versi nonlinear dan linear berbeda (`allclose = False`), sementara output linear
   2-layer identik (`allclose = True`, selisih ~`1e-16`) dengan satu transformasi linear
   tunggal. Ini mengonfirmasi derivasi aljabar secara numerik.

4. **Relevansi untuk LSTM.** Setiap *gate* dalam LSTM — *forget gate* $f_t$, *input
   gate* $i_t$, *output gate* $o_t$ — selalu dibungkus fungsi **sigmoid** ($\sigma$)
   agar nilainya berada di rentang $(0, 1)$ dan bisa berperan sebagai "saklar"/filter
   informasi (0 = tutup penuh, 1 = buka penuh). Sementara kandidat *cell state*
   $\tilde{C}_t$ dibungkus **tanh** agar nilainya berada di $(-1, 1)$, membawa informasi
   yang bisa "menambah" atau "mengurangi" *cell state*. Tanpa nonlinearitas ini, seluruh
   mekanisme *gating* LSTM (yang merupakan inti kemampuannya mengingat/melupakan
   informasi secara selektif dan menangani hubungan non-linear jangka panjang) akan
   runtuh menjadi sekadar kombinasi linear — persis seperti yang kita buktikan pada
   MLP sederhana di notebook ini. Memahami eksperimen di atas adalah fondasi konseptual
   sebelum masuk ke formulasi lengkap gate-gate LSTM.
